In [1]:
import sys

sys.path.append("..")


import matplotlib.pyplot as plt
import numpy as np
import torch

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
from src.data_preparation import load_numpy_files

signal_prefix = f"{DATA_DIR}/sig"
background_prefix = f"{DATA_DIR}/bg"
signal_only_prefix = f"{DATA_DIR}/sig_only"


In [2]:
import src.torch.pre_processing.graph_batching as graph_batching
from importlib import reload
reload(graph_batching)

<module 'src.torch.pre_processing.graph_batching' from '/Users/simi/mu3e_trigger/notebooks_supervised/../src/torch/pre_processing/graph_batching.py'>

In [3]:
train_dataset, test_dataset = graph_batching.create_dataset(
    signal_prefix,
    has_layer_feature=True,
    n_events=30000,
    split=(0.7, 0.3),
    type="hetero",
    whole_event_mode=False,
    timing_cutoff=8,
    mppc_timing_cutoff=2,)


In [4]:
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [5]:
import torch
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv, BatchNorm
from src.torch.model.components import get_mlp


class EdgeClassifier(nn.Module):
    def __init__(self, node_dims, edge_types, num_layers,
                 hidden_dim=64, dropout=0.1):
        super().__init__()

        if not (0.0 <= dropout <= 1.0):
            raise ValueError("Dropout must be in the range [0.0, 1.0].")

        self.dropout = dropout
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Project all node features into a shared hidden_dim space
        self.input_embeddings = nn.ModuleDict({
            node_type: get_mlp(node_dim, hidden_dim, num_layers=2)
            for node_type, node_dim in node_dims.items()
        })

        # Convolutional message passing layers with residuals + batchnorm
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv((hidden_dim, hidden_dim), hidden_dim)
                for edge_type in edge_types
            }, aggr="mean")
            self.convs.append(conv)

            norm = nn.ModuleDict({
                node_type: BatchNorm(hidden_dim)
                for node_type in node_dims
            })
            self.norms.append(norm)

        # Edge-specific classifiers (binary by default)
        self.edge_classifiers = nn.ModuleDict({
            "__".join(edge_type): get_mlp(2 * hidden_dim, 1, num_layers=3)
            for edge_type in edge_types
        })

    def forward(self, hetero_graph: HeteroData):
        x_dict, edge_index_dict = hetero_graph.x_dict, hetero_graph.edge_index_dict

        # Initial node embeddings
        for node_type, x in x_dict.items():
            x = self.input_embeddings[node_type](x)
            x = torch.relu(x)
            x = nn.functional.dropout(x, p=self.dropout, training=self.training)
            x_dict[node_type] = x

        # Convolutional message passing with residuals + norm + dropout
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            new_x_dict = conv(x_dict, edge_index_dict)

            updated_x_dict = {}
            for node_type, x_new in new_x_dict.items():
                x_old = x_dict[node_type]

                # BatchNorm per node type
                x_new = norm[node_type](x_new)

                # Residual connection (only if dimensions match)
                if x_new.shape == x_old.shape:
                    x_new = x_new + x_old

                # Dropout
                x_new = nn.functional.dropout(x_new, p=self.dropout, training=self.training)

                updated_x_dict[node_type] = x_new

            x_dict = updated_x_dict

        # Edge classification
        edge_preds = {}
        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            src_x, dst_x = x_dict[src_type], x_dict[dst_type]

            edge_feat = torch.cat([src_x[edge_index[0]], dst_x[edge_index[1]]], dim=-1)
            edge_pred = self.edge_classifiers["__".join(edge_type)](edge_feat).view(-1)
            edge_preds[edge_type] = edge_pred

        return edge_preds


In [6]:
edge_classifier = EdgeClassifier(
    node_dims=train_dataset.get_node_dims(),
    edge_types=train_dataset.get_edge_types(),
    num_layers=5,
    hidden_dim=64,
    dropout=0.1
)
print(f"Number of parameters: {sum(p.numel() for p in edge_classifier.parameters() if p.requires_grad)}")

Number of parameters: 182180


In [10]:
train_dataset[0]["mppc"]

{'x': tensor([[  54.9054,  -29.3380, -150.1250,    7.1500],
        [  54.9701,  -29.0965, -150.1250,    7.1000],
        [ -46.3114,   39.4293,  150.1250,    7.9500],
        [ -46.4881,   39.2525,  150.1250,    7.6000],
        [  54.9767,  -29.0721,  137.6250,    6.7500]]), 'track_truth': tensor([[ 22.1369, -18.0760,  -6.1254,  29.2330, -11.0000],
        [ 22.1369, -18.0760,  -6.1254,  29.2330, -11.0000],
        [ 22.1369, -18.0760,  -6.1254,  29.2330, -11.0000],
        [ 22.1369, -18.0760,  -6.1254,  29.2330, -11.0000],
        [ 22.1369, -18.0760,  -6.1254,  29.2330, -11.0000]])}

In [7]:
from src.torch.training import FocalLoss, HeteroLossWrapper
from sklearn.metrics import roc_auc_score
criterion = HeteroLossWrapper(FocalLoss(gamma=2.0))
optimizer = torch.optim.AdamW(edge_classifier.parameters(), lr=0.001)

from tqdm import tqdm
num_epochs = 20
for epoch in range(num_epochs):
    edge_classifier.train()
    total_loss = 0
    for hetero_graph in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        out = edge_classifier(hetero_graph)
        # Assuming labels are stored in hetero_graph.y
        loss = criterion(out, hetero_graph.edge_labels_dict)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1/20: 100%|██████████| 210/210 [01:48<00:00,  1.93it/s]


Epoch 1, Loss: 0.1780


Epoch 2/20: 100%|██████████| 210/210 [02:02<00:00,  1.72it/s]


Epoch 2, Loss: 0.1640


Epoch 3/20: 100%|██████████| 210/210 [02:05<00:00,  1.67it/s]


...

Epoch 5, Loss: 0.1596


Epoch 6/20: 100%|██████████| 210/210 [02:07<00:00,  1.65it/s]


Epoch 6, Loss: 0.1591


Epoch 7/20:  59%|█████▉    | 124/210 [01:09<00:48,  1.78it/s]


KeyboardInterrupt: 

In [ ]:
predictions = {}
labels = {}
edge_classifier.eval()
with torch.no_grad():
    for hetero_graph in tqdm(test_loader, desc="Evaluating"):
        out = edge_classifier(hetero_graph)
        for edge_type in out:
            if edge_type not in predictions:
                predictions[edge_type] = []
                labels[edge_type] = []
            predictions[edge_type].append(torch.sigmoid(out[edge_type].cpu()))
            labels[edge_type].append(hetero_graph.edge_labels_dict[edge_type].cpu())
    for edge_type in predictions:
        predictions[edge_type] = torch.cat(predictions[edge_type])
        labels[edge_type] = torch.cat(labels[edge_type])
        auc = roc_auc_score(labels[edge_type].numpy(), predictions[edge_type].numpy())
        print(f"Edge type: {edge_type}, AUC: {auc:.4f}")